In [1]:
# CONSTANTS

INPUT_WB_NAME  = "2026 Stat Arb Mkt Data.xlsx"

FORMULA_TAB = "MKT DATA"

NO_COPY_LIST = ['HI YLD', 'AGG', 'CORPS old', 'MAIN',
                'REAL ESTATE', 'README', 'STATIC DATA', 'MKT DATA']

BIG_RANGE     = "A:T"
SMALL_RANGE   = "C:D"
DATE_RANGE    = "B"
FORMULA_RANGE = "Q1:R1"


In [2]:
# import pandas as pd

from datetime import date, datetime, timedelta
import pandas_market_calendars as mcal


In [3]:
# --- system setup ---
import os
import sys
sys.path.append(os.path.abspath(".."))

In [4]:
# --- utils ---
# from IPython.display import display, clear_output
# from input_output.Standard_Input_and_Output import standard_input, standard_output
import xlwings as xw
from input_output.Class_InputOutput import InputOutput
io = InputOutput()

In [5]:

_NYSE = mcal.get_calendar("NYSE")   # module-level, built once

def next_nyse_trading_day(date_):
    if date_ is not None:
        
        # Convert datetime/Timestamp to date
        if isinstance(date_, datetime):
            date_ = date_.date()
        elif hasattr(date_, "date") and not isinstance(date_, date):
            date_ = date_.date()

        schedule = _NYSE.valid_days(start_date=date_, end_date=date_ + timedelta(days=10))
        return schedule[schedule.date > date_].min().date()
    return None    

In [6]:
wb = io.set_xw_book(INPUT_WB_NAME)

tab_names = [sheet.name for sheet in wb.sheets]

tab_names = [
    tab_name
    for tab_name in tab_names
    if tab_name not in NO_COPY_LIST
]

base_ws = io.set_xw_sheet(wb, tab_names[0])

last_row = base_ws.range(f"A{base_ws.cells.last_cell.row}").end("up").row 
next_row = last_row + 1

last_range = BIG_RANGE.replace(":", f"{last_row}:") + str(last_row)
next_range = BIG_RANGE.replace(":", f"{next_row}:") + str(next_row)
base_ws.range(last_range).copy(destination=base_ws.range(next_range))

last_date = base_ws.range(DATE_RANGE + str(last_row)).value
next_date = next_nyse_trading_day(last_date)
next_date_range = base_ws.range(DATE_RANGE + str(next_row))
next_date_range.value = next_date

formula_ws = io.set_xw_sheet(wb, FORMULA_TAB)
current_formula_range = formula_ws.range(FORMULA_RANGE)
small_range = SMALL_RANGE.replace(":", f"{next_row}:") + str(next_row)

new_formula_range = base_ws.range(small_range)
new_formula_range.formula = current_formula_range.formula


In [ ]:
for tab in tab_names[1:]:
    next_ws = io.set_xw_sheet(wb, tab)
    base_ws.range(next_range).copy(destination=next_ws.range(next_range))
    next_ws.range(next_range).value = next_ws.range(next_range).value

base_ws.range(next_range).value = base_ws.range(next_range).value